<a href="https://colab.research.google.com/github/Sophiajackrich/University-of-Oulu/blob/Pbeginners/DEEP_LEARNING_FINAL_PROJECTT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#           MULTI-RETINAL DISEASE DETECTION DEEP LEARNING PROJECT



In [1]:
import os
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score

#Additionalmechanism import
from torchvision.models import swin_t, Swin_T_Weights



In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:

# LOADING DATASET

class RetinaMultiLabelDataset(Dataset):
    """
    CSV format expected: [id, D, G, A]
    image_dir contains the images named by the id column.
    """
    def __init__(self, csv_file: str, image_dir: str, transform=None):
        self.data = pd.read_csv(csv_file)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        img_path = os.path.join(self.image_dir, row.iloc[0])
        img = Image.open(img_path).convert("RGB")
        labels = torch.tensor(row[1:].values.astype("float32"))
        if self.transform:
            img = self.transform(img)
        return img, labels



# FUNCTION FOR MAKING TRANSFORMS

def make_transforms(img_size=256, train_aug=False):
    if train_aug:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(0.2, 0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])


# Function for Metrics/Evaluation

def evaluate_offsite(model, loader, device, threshold=0.5):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            logits = model(imgs)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs > threshold).astype(int)
            y_true.append(labels.numpy())
            y_pred.append(preds)

    y_true = np.concatenate(y_true, axis=0)
    y_pred = np.concatenate(y_pred, axis=0)

    diseases = ["DR", "Glaucoma", "AMD"]
    for i, d in enumerate(diseases):
        yt, yp = y_true[:, i], y_pred[:, i]
        acc = accuracy_score(yt, yp)
        prec = precision_score(yt, yp, zero_division=0)
        rec = recall_score(yt, yp, zero_division=0)
        f1 = f1_score(yt, yp, zero_division=0)
        kappa = cohen_kappa_score(yt, yp)
        print(f"{d} -> Acc: {acc:.4f} | Prec: {prec:.4f} | Rec: {rec:.4f} | F1: {f1:.4f} | Kappa: {kappa:.4f}")



# Task 1.1 — Function for Building no fine-tuning model

def build_model_no_finetune(backbone: str, pretrained_path: str):
    if backbone == "efficientnet":
        model = models.efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, 3)

    elif backbone == "resnet18":
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, 3)

    else:
        raise ValueError("Unsupported backbone for Task 1.1")

    state = torch.load(pretrained_path, map_location="cpu")
    model.load_state_dict(state, strict=True)

    for p in model.parameters():
        p.requires_grad = False

    return model


def evaluate_no_finetuning(
    backbone: str,
    test_csv: str,
    test_image_dir: str,
    pretrained_path: str,
    batch_size: int = 32,
    img_size: int = 256,
    threshold: float = 0.5
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n=== Task 1.1 (No fine-tuning) | backbone={backbone} | device={device} ===\n")

    tf = make_transforms(img_size, train_aug=False)
    test_ds = RetinaMultiLabelDataset(test_csv, test_image_dir, tf)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    model = build_model_no_finetune(backbone, pretrained_path).to(device)
    evaluate_offsite(model, test_loader, device=device, threshold=threshold)


def generate_kaggle_submission_no_finetune(
    backbone: str,
    onsite_csv: str,
    onsite_img_dir: str,
    pretrained_path: str,
    output_csv: str = "task1_1_submission.csv",
    batch_size: int = 32,
    img_size: int = 256,
    threshold: float = 0.5
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n=== Generating Task 1.1 Kaggle Submission | backbone={backbone} | device={device} ===\n")

    df = pd.read_csv(onsite_csv)
    image_names = df.iloc[:, 0].values

    tf = make_transforms(img_size, train_aug=False)

    class OnsiteDataset(Dataset):
        def __init__(self, names, img_dir):
            self.names = names
            self.img_dir = img_dir

        def __len__(self):
            return len(self.names)

        def __getitem__(self, idx):
            name = self.names[idx]
            img = Image.open(os.path.join(self.img_dir, name)).convert("RGB")
            return tf(img), name

    loader = DataLoader(OnsiteDataset(image_names, onsite_img_dir),
                        batch_size=batch_size, shuffle=False, num_workers=2)

    model = build_model_no_finetune(backbone, pretrained_path).to(device)
    model.eval()

    rows = []
    with torch.no_grad():
        for imgs, names in loader:
            imgs = imgs.to(device)
            logits = model(imgs)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs > threshold).astype(int)
            for n, p in zip(names, preds):
                rows.append([n, int(p[0]), int(p[1]), int(p[2])])

    sub = pd.DataFrame(rows, columns=["id", "D", "G", "A"])
    sub.to_csv(output_csv, index=False, sep=",")
    print(f"Saved -> {output_csv}")
    print(sub.head())
    return output_csv



# Task 2.1 — Focal Loss Class

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.gamma = float(gamma)
        self.reduction = reduction

        if alpha is None:
            self.alpha = None
        else:
            if isinstance(alpha, (list, tuple, np.ndarray)):
                self.alpha = torch.tensor(alpha, dtype=torch.float32)
            elif isinstance(alpha, torch.Tensor):
                self.alpha = alpha.float()
            else:
                self.alpha = torch.tensor([float(alpha)], dtype=torch.float32)

    def forward(self, logits, targets):
        bce = torch.nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction="none"
        )
        p = torch.sigmoid(logits)
        p_t = p * targets + (1 - p) * (1 - targets)
        focal_factor = (1 - p_t) ** self.gamma
        loss = focal_factor * bce

        if self.alpha is not None:
            alpha = self.alpha.to(logits.device)
            if alpha.numel() == 1:
                alpha = alpha.repeat(logits.size(1))
            alpha_t = alpha.unsqueeze(0) * targets + (1 - alpha.unsqueeze(0)) * (1 - targets)
            loss = alpha_t * loss

        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss

# Task 2.2 — Class-balanced BCE pos_weight
def add_pos_weight_from_csv(train_csv: str, num_classes: int = 3, eps: float = 1e-6):
    df = pd.read_csv(train_csv)
    y = df.iloc[:, 1:1 + num_classes].values.astype(np.float32)
    pos = y.sum(axis=0)
    neg = (y == 0).sum(axis=0)
    pos_weight = neg / (pos + eps)
    return torch.tensor(pos_weight, dtype=torch.float32)

# Task 3.1 — SE Block + EfficientNet Wrapper Class
class SEBlock(nn.Module):
    def __init__(self, channels: int, reduction: int = 16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden, bias=True),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels, bias=True),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.shape
        s = self.pool(x).view(b, c)
        e = self.fc(s).view(b, c, 1, 1)
        return x * e

class EfficientNetB0_WithExtraSE(nn.Module):
    def __init__(self, num_classes: int = 3, se_reduction: int = 16):
        super().__init__()
        base = models.efficientnet_b0(weights=None)
        self.features = base.features
        self.out_channels = 1280
        self.se = SEBlock(self.out_channels, reduction=se_reduction)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(self.out_channels, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = self.se(x)
        x = self.pool(x).flatten(1)
        x = self.classifier(x)
        return x



# Task 3.2 — MHA Wrapper
class EfficientNetB0_WithMHA(nn.Module):
    def __init__(self, num_classes: int = 3, num_heads: int = 8, attn_dropout: float = 0.0):
        super().__init__()
        base = models.efficientnet_b0(weights=None)
        self.features = base.features
        self.embed_dim = 1280

        self.mha = nn.MultiheadAttention(
            embed_dim=self.embed_dim,
            num_heads=num_heads,
            dropout=attn_dropout,
            batch_first=True
        )
        self.norm1 = nn.LayerNorm(self.embed_dim)
        self.norm2 = nn.LayerNorm(self.embed_dim)

        self.ffn = nn.Sequential(
            nn.Linear(self.embed_dim, self.embed_dim * 2),
            nn.GELU(),
            nn.Dropout(attn_dropout),
            nn.Linear(self.embed_dim * 2, self.embed_dim),
        )
        self.dropout = nn.Dropout(attn_dropout)
        self.classifier = nn.Linear(self.embed_dim, num_classes)

    def forward(self, x):
        x = self.features(x)
        b, c, h, w = x.shape
        tokens = x.permute(0, 2, 3, 1).reshape(b, h * w, c)

        t0 = self.norm1(tokens)
        attn_out, _ = self.mha(t0, t0, t0, need_weights=False)
        tokens = tokens + self.dropout(attn_out)

        t1 = self.norm2(tokens)
        ffn_out = self.ffn(t1)
        tokens = tokens + self.dropout(ffn_out)

        pooled = tokens.mean(dim=1)
        logits = self.classifier(pooled)
        return logits



# Task 4.1 — Swin Tiny builder (MISSING in your pasted code)
def build_swin_tiny(num_classes: int = 3, imagenet_pretrained: bool = True):
    """
    Swin Transformer Tiny backbone from torchvision.
    Output head is replaced for 3-label logits (multi-label).
    """
    weights = Swin_T_Weights.IMAGENET1K_V1 if imagenet_pretrained else None
    model = swin_t(weights=weights)
    in_features = model.head.in_features
    model.head = nn.Linear(in_features, num_classes)
    return model



# Build model from PROVIDED pretrained checkpoint (for tasks 1.2 and above)
def build_from_provided_checkpoint(
    backbone: str,
    pretrained_backbone_path: str,
    num_classes: int = 3,
    use_se: bool = False,
    se_reduction: int = 16,
    use_mha: bool = False,
    mha_heads: int = 8,
    mha_dropout: float = 0.0
):
    if backbone == "resnet18":
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        state = torch.load(pretrained_backbone_path, map_location="cpu")
        model.load_state_dict(state, strict=True)
        return model

    if backbone != "efficientnet":
        raise ValueError("backbone must be 'resnet18' or 'efficientnet'")

    base = models.efficientnet_b0(weights=None)
    base.classifier[1] = nn.Linear(base.classifier[1].in_features, num_classes)
    state = torch.load(pretrained_backbone_path, map_location="cpu")
    base.load_state_dict(state, strict=True)

    if use_se:
        m = EfficientNetB0_WithExtraSE(num_classes=num_classes, se_reduction=se_reduction)
        m.features.load_state_dict(base.features.state_dict(), strict=True)
        return m

    if use_mha:
        m = EfficientNetB0_WithMHA(num_classes=num_classes, num_heads=mha_heads, attn_dropout=mha_dropout)
        m.features.load_state_dict(base.features.state_dict(), strict=True)
        return m

    normal = models.efficientnet_b0(weights=None)
    normal.classifier[1] = nn.Linear(normal.classifier[1].in_features, num_classes)
    normal.load_state_dict(state, strict=True)
    return normal



# Load model for inference (submissions for tasks 1.2 and above)
def load_model_for_inference(
    backbone: str,
    ckpt_path: str,
    num_classes: int = 3,
    device="cpu",
    use_se: bool = False,
    se_reduction: int = 16,
    use_mha: bool = False,
    mha_heads: int = 8,
    mha_dropout: float = 0.0,
    imagenet_pretrained: bool = False
):
    if backbone == "resnet18":
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)

    elif backbone == "efficientnet":
        if use_se:
            model = EfficientNetB0_WithExtraSE(num_classes=num_classes, se_reduction=se_reduction)
        elif use_mha:
            model = EfficientNetB0_WithMHA(num_classes=num_classes, num_heads=mha_heads, attn_dropout=mha_dropout)
        else:
            model = models.efficientnet_b0(weights=None)
            model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)

    elif backbone == "swin_t":
        model = build_swin_tiny(num_classes=num_classes, imagenet_pretrained=imagenet_pretrained)

    else:
        raise ValueError("Unsupported backbone")

    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state, strict=True)
    return model


# Kaggle submission generator (tasks 1.2 and above)
def generate_kaggle_submission(
    backbone: str,
    ckpt_path: str,
    onsite_csv: str,
    onsite_img_dir: str,
    output_csv: str,
    batch_size: int = 32,
    img_size: int = 256,
    threshold: float = 0.5,
    use_se: bool = False,
    se_reduction: int = 16,
    use_mha: bool = False,
    mha_heads: int = 8,
    mha_dropout: float = 0.0
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(
        f"\n=== Generating Kaggle Submission | backbone={backbone} | "
        f"use_se={use_se} | use_mha={use_mha} | ckpt={ckpt_path} ===\n"
    )

    df = pd.read_csv(onsite_csv)
    image_names = df.iloc[:, 0].values

    tf = make_transforms(img_size, train_aug=False)

    class OnsiteDataset(Dataset):
        def __init__(self, names, img_dir):
            self.names = names
            self.img_dir = img_dir

        def __len__(self):
            return len(self.names)

        def __getitem__(self, idx):
            name = self.names[idx]
            img = Image.open(os.path.join(self.img_dir, name)).convert("RGB")
            return tf(img), name

    loader = DataLoader(OnsiteDataset(image_names, onsite_img_dir),
                        batch_size=batch_size, shuffle=False, num_workers=2)

    model = load_model_for_inference(
        backbone=backbone,
        ckpt_path=ckpt_path,
        num_classes=3,
        device=device,
        use_se=use_se,
        se_reduction=se_reduction,
        use_mha=use_mha,
        mha_heads=mha_heads,
        mha_dropout=mha_dropout
    ).to(device)

    model.eval()
    rows = []
    with torch.no_grad():
        for imgs, names in loader:
            imgs = imgs.to(device)
            logits = model(imgs)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs > threshold).astype(int)
            for n, p in zip(names, preds):
                rows.append([n, int(p[0]), int(p[1]), int(p[2])])

    sub = pd.DataFrame(rows, columns=["id", "D", "G", "A"])
    sub.to_csv(output_csv, index=False, sep=",")
    print(f"Saved -> {output_csv}")
    print(sub.head())
    return output_csv



# Task 4.3 - ENSEMBLE GENERATOR FOR KAGGLE SUBMISSION
def generate_kaggle_submission_ensemble(
    models_spec: list,
    onsite_csv: str,
    onsite_img_dir: str,
    output_csv: str,
    batch_size: int = 32,
    img_size: int = 256,
    threshold: float = 0.5,
    weights: list | None = None
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    df = pd.read_csv(onsite_csv)
    image_names = df.iloc[:, 0].values

    tf = make_transforms(img_size, train_aug=False)

    class OnsiteDataset(Dataset):
        def __init__(self, names, img_dir):
            self.names = names
            self.img_dir = img_dir

        def __len__(self):
            return len(self.names)

        def __getitem__(self, idx):
            name = self.names[idx]
            img = Image.open(os.path.join(self.img_dir, name)).convert("RGB")
            return tf(img), name

    loader = DataLoader(OnsiteDataset(image_names, onsite_img_dir),
                        batch_size=batch_size, shuffle=False, num_workers=2)

    if weights is None:
        weights = [1.0] * len(models_spec)
    if len(weights) != len(models_spec):
        raise ValueError("weights must match models_spec length")

    w = np.array(weights, dtype=np.float32)
    w = w / w.sum()

    ensemble_models = []
    for spec in models_spec:
        m = load_model_for_inference(
            backbone=spec["backbone"],
            ckpt_path=spec["ckpt_path"],
            num_classes=3,
            device=device,
            use_se=spec.get("use_se", False),
            se_reduction=spec.get("se_reduction", 16),
            use_mha=spec.get("use_mha", False),
            mha_heads=spec.get("mha_heads", 8),
            mha_dropout=spec.get("mha_dropout", 0.0),
            imagenet_pretrained=False
        ).to(device)
        m.eval()
        ensemble_models.append(m)

    rows = []
    with torch.no_grad():
        for imgs, names in loader:
            imgs = imgs.to(device)

            prob_sum = None
            for mi, m in enumerate(ensemble_models):
                logits = m(imgs)
                probs = torch.sigmoid(logits)
                if prob_sum is None:
                    prob_sum = w[mi] * probs
                else:
                    prob_sum = prob_sum + w[mi] * probs

            probs_np = prob_sum.cpu().numpy()
            preds = (probs_np > threshold).astype(int)

            for n, p in zip(names, preds):
                rows.append([n, int(p[0]), int(p[1]), int(p[2])])

    sub = pd.DataFrame(rows, columns=["id", "D", "G", "A"])
    sub.to_csv(output_csv, index=False, sep=",")
    print(f"Saved ensemble submission -> {output_csv}")
    print(sub.head())
    return output_csv



# Training loop helper Function
def _train_loop(model, train_loader, val_loader, device, criterion, optimizer, epochs, ckpt_path):
    best_val_loss = float("inf")

    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = 0.0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            logits = model(imgs)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * imgs.size(0)

        train_loss /= len(train_loader.dataset)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                logits = model(imgs)
                loss = criterion(logits, labels)
                val_loss += loss.item() * imgs.size(0)

        val_loss /= len(val_loader.dataset)
        print(f"Epoch {epoch}/{epochs} | TrainLoss={train_loss:.4f} | ValLoss={val_loss:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), ckpt_path)
            print(f"  ✔ Saved best -> {ckpt_path}")



# Task 1.2 — Frozen backbone Training Function
def train_task12_frozen_backbone(
    backbone: str,
    train_csv: str, val_csv: str, test_csv: str,
    train_img_dir: str, val_img_dir: str, test_img_dir: str,
    pretrained_backbone_path: str,
    epochs=20, batch_size=32, lr=1e-4, img_size=256,
    save_dir="/content/checkpoints"
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n=== Task 1.2 (Frozen backbone) | {backbone} | device={device} ===\n")

    train_tf = make_transforms(img_size, train_aug=True)
    eval_tf = make_transforms(img_size, train_aug=False)

    train_ds = RetinaMultiLabelDataset(train_csv, train_img_dir, train_tf)
    val_ds = RetinaMultiLabelDataset(val_csv, val_img_dir, eval_tf)
    test_ds = RetinaMultiLabelDataset(test_csv, test_img_dir, eval_tf)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    model = build_from_provided_checkpoint(backbone, pretrained_backbone_path, num_classes=3).to(device)

    for p in model.parameters():
        p.requires_grad = False

    if backbone == "resnet18":
        for p in model.fc.parameters():
            p.requires_grad = True
        trainable_params = model.fc.parameters()
    else:
        for p in model.classifier[1].parameters():
            p.requires_grad = True
        trainable_params = model.classifier[1].parameters()

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(trainable_params, lr=lr)

    os.makedirs(save_dir, exist_ok=True)
    ckpt_path = os.path.join(save_dir, f"best_frozen_{backbone}.pt")

    _train_loop(model, train_loader, val_loader, device, criterion, optimizer, epochs, ckpt_path)

    print("\n=== Task 1.2 Offsite Test Metrics ===")
    model.load_state_dict(torch.load(ckpt_path, map_location=device), strict=True)
    evaluate_offsite(model, test_loader, device=device)

    return ckpt_path



# Task 1.3 — Full fine-tune - Trainin Function
def train_task13_full_finetune(
    backbone: str,
    train_csv: str, val_csv: str, test_csv: str,
    train_img_dir: str, val_img_dir: str, test_img_dir: str,
    pretrained_backbone_path: str,
    epochs=20, batch_size=32, lr=1e-4, img_size=256,
    save_dir="/content/checkpoints"
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n=== Task 1.3 (Full fine-tune) | {backbone} | device={device} ===\n")

    train_tf = make_transforms(img_size, train_aug=True)
    eval_tf = make_transforms(img_size, train_aug=False)

    train_ds = RetinaMultiLabelDataset(train_csv, train_img_dir, train_tf)
    val_ds = RetinaMultiLabelDataset(val_csv, val_img_dir, eval_tf)
    test_ds = RetinaMultiLabelDataset(test_csv, test_img_dir, eval_tf)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    model = build_from_provided_checkpoint(backbone, pretrained_backbone_path, num_classes=3).to(device)
    for p in model.parameters():
        p.requires_grad = True

    pos_weight = add_pos_weight_from_csv(train_csv, num_classes=3).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    os.makedirs(save_dir, exist_ok=True)
    ckpt_path = os.path.join(save_dir, f"best_full_{backbone}.pt")

    _train_loop(model, train_loader, val_loader, device, criterion, optimizer, epochs, ckpt_path)

    print("\n=== Task 1.3 Offsite Test Metrics ===")
    model.load_state_dict(torch.load(ckpt_path, map_location=device), strict=True)
    evaluate_offsite(model, test_loader, device=device)

    return ckpt_path


# Task 2.1 — Full fine-tune with Focal Loss -- Trainin Function
def train_task21_focal_full_finetune(
    backbone: str,
    train_csv: str, val_csv: str, test_csv: str,
    train_img_dir: str, val_img_dir: str, test_img_dir: str,
    pretrained_backbone_path: str,
    epochs=20, batch_size=32, lr=1e-4, img_size=256,
    save_dir="/content/checkpoints",
    gamma=2.0,
    alpha=None
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n=== Task 2.1 (Focal Loss Full FT) | {backbone} | device={device} ===\n")

    train_tf = make_transforms(img_size, train_aug=True)
    eval_tf = make_transforms(img_size, train_aug=False)

    train_ds = RetinaMultiLabelDataset(train_csv, train_img_dir, train_tf)
    val_ds = RetinaMultiLabelDataset(val_csv, val_img_dir, eval_tf)
    test_ds = RetinaMultiLabelDataset(test_csv, test_img_dir, eval_tf)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    model = build_from_provided_checkpoint(backbone, pretrained_backbone_path, num_classes=3).to(device)
    for p in model.parameters():
        p.requires_grad = True

    criterion = FocalLoss(alpha=alpha, gamma=gamma, reduction="mean").to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    os.makedirs(save_dir, exist_ok=True)
    ckpt_path = os.path.join(save_dir, f"best_task21_focal_full_{backbone}.pt")

    _train_loop(model, train_loader, val_loader, device, criterion, optimizer, epochs, ckpt_path)

    print("\n=== Task 2.1 Offsite Test Metrics ===")
    model.load_state_dict(torch.load(ckpt_path, map_location=device), strict=True)
    evaluate_offsite(model, test_loader, device=device)

    return ckpt_path

# Task 2.2 — Full fine-tune with Class-Balanced BCE -- Trainin Function
def train_task22_classbalanced_full_finetune(
    backbone: str,
    train_csv: str, val_csv: str, test_csv: str,
    train_img_dir: str, val_img_dir: str, test_img_dir: str,
    pretrained_backbone_path: str,
    epochs=20, batch_size=32, lr=1e-4, img_size=256,
    save_dir="/content/checkpoints"
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n=== Task 2.2 (Class-Balanced BCE Full FT) | {backbone} | device={device} ===\n")

    train_tf = make_transforms(img_size, train_aug=True)
    eval_tf = make_transforms(img_size, train_aug=False)

    train_ds = RetinaMultiLabelDataset(train_csv, train_img_dir, train_tf)
    val_ds = RetinaMultiLabelDataset(val_csv, val_img_dir, eval_tf)
    test_ds = RetinaMultiLabelDataset(test_csv, test_img_dir, eval_tf)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    model = build_from_provided_checkpoint(backbone, pretrained_backbone_path, num_classes=3).to(device)
    for p in model.parameters():
        p.requires_grad = True

    pos_weight = add_pos_weight_from_csv(train_csv, num_classes=3).to(device)
    print("pos_weight (D,G,A):", pos_weight.detach().cpu().numpy())

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    os.makedirs(save_dir, exist_ok=True)
    ckpt_path = os.path.join(save_dir, f"best_task22_cb_full_{backbone}.pt")

    _train_loop(model, train_loader, val_loader, device, criterion, optimizer, epochs, ckpt_path)

    print("\n=== Task 2.2 Offsite Test Metrics ===")
    model.load_state_dict(torch.load(ckpt_path, map_location=device), strict=True)
    evaluate_offsite(model, test_loader, device=device)

    return ckpt_path

# Task 3.1 — EfficientNet + SE + Focal Loss -- Trainin Function
def train_task31_se_focal_full_finetune(
    train_csv: str, val_csv: str, test_csv: str,
    train_img_dir: str, val_img_dir: str, test_img_dir: str,
    pretrained_efficientnet_path: str,
    epochs=20, batch_size=32, lr=1e-4, img_size=256,
    save_dir="/content/checkpoints",
    gamma=2.0,
    alpha=None,
    se_reduction: int = 16
):
    backbone = "efficientnet"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n=== Task 3.1 (EfficientNet + SE + Focal) | device={device} ===\n")

    train_tf = make_transforms(img_size, train_aug=True)
    eval_tf = make_transforms(img_size, train_aug=False)

    train_ds = RetinaMultiLabelDataset(train_csv, train_img_dir, train_tf)
    val_ds = RetinaMultiLabelDataset(val_csv, val_img_dir, eval_tf)
    test_ds = RetinaMultiLabelDataset(test_csv, test_img_dir, eval_tf)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    model = build_from_provided_checkpoint(
        backbone=backbone,
        pretrained_backbone_path=pretrained_efficientnet_path,
        num_classes=3,
        use_se=True,
        se_reduction=se_reduction
    ).to(device)

    for p in model.parameters():
        p.requires_grad = True

    criterion = FocalLoss(alpha=alpha, gamma=gamma, reduction="mean").to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    os.makedirs(save_dir, exist_ok=True)
    ckpt_path = os.path.join(save_dir, "best_task31_se_focal_efficientnet.pt")

    _train_loop(model, train_loader, val_loader, device, criterion, optimizer, epochs, ckpt_path)

    print("\n=== Task 3.1 Offsite Test Metrics ===")
    model.load_state_dict(torch.load(ckpt_path, map_location=device), strict=True)
    evaluate_offsite(model, test_loader, device=device)

    return ckpt_path

# Task 3.2 — EfficientNet + MHA + Focal Loss -- Trainin Function
def train_task32_mha_focal_full_finetune(
    train_csv: str, val_csv: str, test_csv: str,
    train_img_dir: str, val_img_dir: str, test_img_dir: str,
    pretrained_efficientnet_path: str,
    epochs=20, batch_size=32, lr=1e-4, img_size=256,
    save_dir="/content/checkpoints",
    gamma=2.0,
    alpha=None,
    mha_heads: int = 8,
    mha_dropout: float = 0.0
):
    backbone = "efficientnet"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n=== Task 3.2 (EfficientNet + MHA + Focal) | heads={mha_heads} | device={device} ===\n")

    train_tf = make_transforms(img_size, train_aug=True)
    eval_tf = make_transforms(img_size, train_aug=False)

    train_ds = RetinaMultiLabelDataset(train_csv, train_img_dir, train_tf)
    val_ds = RetinaMultiLabelDataset(val_csv, val_img_dir, eval_tf)
    test_ds = RetinaMultiLabelDataset(test_csv, test_img_dir, eval_tf)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=2)

    model = build_from_provided_checkpoint(
        backbone=backbone,
        pretrained_backbone_path=pretrained_efficientnet_path,
        num_classes=3,
        use_mha=True,
        mha_heads=mha_heads,
        mha_dropout=mha_dropout
    ).to(device)

    for p in model.parameters():
        p.requires_grad = True

    criterion = FocalLoss(alpha=alpha, gamma=gamma, reduction="mean").to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    os.makedirs(save_dir, exist_ok=True)
    ckpt_path = os.path.join(save_dir, "best_task32_mha_focal_efficientnet.pt")

    _train_loop(model, train_loader, val_loader, device, criterion, optimizer, epochs, ckpt_path)

    print("\n=== Task 3.2 Offsite Test Metrics ===")
    model.load_state_dict(torch.load(ckpt_path, map_location=device), strict=True)
    evaluate_offsite(model, test_loader, device=device)

    return ckpt_path

# Task 4.1 — SWIN TRAINING FUNCTION -- Trainin Function
def train_task41_swin_focal_full_finetune(
    train_csv: str, val_csv: str, test_csv: str,
    train_img_dir: str, val_img_dir: str, test_img_dir: str,
    epochs: int = 20,
    batch_size: int = 32,
    lr: float = 1e-4,
    img_size: int = 224,
    save_dir: str = "/content/checkpoints",
    gamma: float = 2.0,
    alpha=None,
    imagenet_pretrained: bool = True
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n=== Task 4.1 (Swin-Tiny + Focal) | device={device} ===\n")

    train_tf = make_transforms(img_size, train_aug=True)
    eval_tf  = make_transforms(img_size, train_aug=False)

    train_ds = RetinaMultiLabelDataset(train_csv, train_img_dir, train_tf)
    val_ds   = RetinaMultiLabelDataset(val_csv,   val_img_dir,   eval_tf)
    test_ds  = RetinaMultiLabelDataset(test_csv,  test_img_dir,  eval_tf)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=2)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=2)

    model = build_swin_tiny(num_classes=3, imagenet_pretrained=imagenet_pretrained).to(device)

    for p in model.parameters():
        p.requires_grad = True

    criterion = FocalLoss(alpha=alpha, gamma=gamma, reduction="mean").to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    os.makedirs(save_dir, exist_ok=True)
    ckpt_path = os.path.join(save_dir, "best_task41_swin_focal.pt")

    _train_loop(model, train_loader, val_loader, device, criterion, optimizer, epochs, ckpt_path)

    print("\n=== Task 4.1 Offsite Test Metrics ===")
    model.load_state_dict(torch.load(ckpt_path, map_location=device), strict=True)
    evaluate_offsite(model, test_loader, device=device)

    return ckpt_path
#-------------------------------------------------------------------------------------------------------------------------------------------------------

# MAIN BLOCK - RUNNING THE PIPELINE FUNCTIONS TO PRODUCE RESULTS
if __name__ == "__main__":

    # Choose backbone for tasks
    backbone = "efficientnet"  # "resnet18" or "efficientnet"

    # Choose task
    # Training/Eval: "1.1","1.2","1.3","2.1","2.2","3.1","3.2","4.1"
    # Submission:    "1.1_submit","1.2_submit","1.3_submit","2.1_submit","2.2_submit","3.1_submit","3.2_submit","4.1_submit","4.3_submit"
    task = "1.1"

    # Paths
    train_csv = "/content/drive/MyDrive/final_project_resources/train.csv"
    val_csv   = "/content/drive/MyDrive/final_project_resources/val.csv"
    test_csv  = "/content/drive/MyDrive/final_project_resources/offsite_test.csv"

    train_img_dir = "/content/drive/MyDrive/final_project_resources/images/train"
    val_img_dir   = "/content/drive/MyDrive/final_project_resources/images/val"
    test_img_dir  = "/content/drive/MyDrive/final_project_resources/images/offsite_test"

    onsite_csv     = "/content/drive/MyDrive/final_project_resources/onsite_test_submission.csv"
    onsite_img_dir = "/content/drive/MyDrive/final_project_resources/images/onsite_test"

    provided_resnet_ckpt = "/content/drive/MyDrive/final_project_resources/pretrained_backbone/ckpt_resnet18_ep50.pt"
    provided_effnet_ckpt = "/content/drive/MyDrive/final_project_resources/pretrained_backbone/ckpt_efficientnet_ep50.pt"

    provided_pretrained_ckpt = provided_resnet_ckpt if backbone == "resnet18" else provided_effnet_ckpt

# --------------------------------------------------------------------------------------------------------------------
    # Task 1.1 (eval) and 1.1_submit

    if task == "1.1":
        evaluate_no_finetuning(
            backbone=backbone,
            test_csv=test_csv,
            test_image_dir=test_img_dir,
            pretrained_path=provided_pretrained_ckpt,
            batch_size=32,
            img_size=256,
            threshold=0.5
        )

    elif task == "1.1_submit":
        generate_kaggle_submission_no_finetune(
            backbone=backbone,
            onsite_csv=onsite_csv,
            onsite_img_dir=onsite_img_dir,
            pretrained_path=provided_pretrained_ckpt,
            output_csv=f"/content/task1_1_submission_{backbone}.csv",
            batch_size=32,
            img_size=256,
            threshold=0.5
        )

#--------------------------------------------------------------------------------------------
    # Task 1.2 / 1.3 / 2.1 / 2.2
    elif task == "1.2":
        train_task12_frozen_backbone(
            backbone=backbone,
            train_csv=train_csv, val_csv=val_csv, test_csv=test_csv,
            train_img_dir=train_img_dir, val_img_dir=val_img_dir, test_img_dir=test_img_dir,
            pretrained_backbone_path=provided_pretrained_ckpt
        )

    elif task == "1.3":
        train_task13_full_finetune(
            backbone=backbone,
            train_csv=train_csv, val_csv=val_csv, test_csv=test_csv,
            train_img_dir=train_img_dir, val_img_dir=val_img_dir, test_img_dir=test_img_dir,
            pretrained_backbone_path=provided_pretrained_ckpt
        )

    elif task == "2.1":
        train_task21_focal_full_finetune(
            backbone=backbone,
            train_csv=train_csv, val_csv=val_csv, test_csv=test_csv,
            train_img_dir=train_img_dir, val_img_dir=val_img_dir, test_img_dir=test_img_dir,
            pretrained_backbone_path=provided_pretrained_ckpt
        )

    elif task == "2.2":
        train_task22_classbalanced_full_finetune(
            backbone=backbone,
            train_csv=train_csv, val_csv=val_csv, test_csv=test_csv,
            train_img_dir=train_img_dir, val_img_dir=val_img_dir, test_img_dir=test_img_dir,
            pretrained_backbone_path=provided_pretrained_ckpt
        )

  #--------------------------------------------------------------------------------------------
    # Task 3.1 / 3.2 (EfficientNet only)

    elif task == "3.1":
        train_task31_se_focal_full_finetune(
            train_csv=train_csv, val_csv=val_csv, test_csv=test_csv,
            train_img_dir=train_img_dir, val_img_dir=val_img_dir, test_img_dir=test_img_dir,
            pretrained_efficientnet_path=provided_effnet_ckpt
        )

    elif task == "3.2":
        train_task32_mha_focal_full_finetune(
            train_csv=train_csv, val_csv=val_csv, test_csv=test_csv,
            train_img_dir=train_img_dir, val_img_dir=val_img_dir, test_img_dir=test_img_dir,
            pretrained_efficientnet_path=provided_effnet_ckpt
        )

#--------------------------------------------------------------------------------------------
    # Task 4.1 (Swin)
    elif task == "4.1":
        train_task41_swin_focal_full_finetune(
            train_csv=train_csv, val_csv=val_csv, test_csv=test_csv,
            train_img_dir=train_img_dir, val_img_dir=val_img_dir, test_img_dir=test_img_dir,
            epochs=20, batch_size=32, lr=1e-4, img_size=224,
            save_dir="/content/checkpoints",
            gamma=2.0,
            alpha=None,
            imagenet_pretrained=True
        )

#--------------------------------------------------------------------------------------------
    # Submission tasks for 1.2+ (trained checkpoints)
    elif task == "1.2_submit":
        generate_kaggle_submission(
            backbone=backbone,
            ckpt_path=f"/content/checkpoints/best_frozen_{backbone}.pt",
            onsite_csv=onsite_csv,
            onsite_img_dir=onsite_img_dir,
            output_csv=f"/content/task1_2_submission_{backbone}.csv"
        )

    elif task == "1.3_submit":
        generate_kaggle_submission(
            backbone=backbone,
            ckpt_path=f"/content/checkpoints/best_full_{backbone}.pt",
            onsite_csv=onsite_csv,
            onsite_img_dir=onsite_img_dir,
            output_csv=f"/content/task1_3_submission_{backbone}.csv"
        )

    elif task == "2.1_submit":
        generate_kaggle_submission(
            backbone=backbone,
            ckpt_path=f"/content/checkpoints/best_task21_focal_full_{backbone}.pt",
            onsite_csv=onsite_csv,
            onsite_img_dir=onsite_img_dir,
            output_csv=f"/content/task2_1_focal_submission_{backbone}.csv"
        )

    elif task == "2.2_submit":
        generate_kaggle_submission(
            backbone=backbone,
            ckpt_path=f"/content/checkpoints/best_task22_cb_full_{backbone}.pt",
            onsite_csv=onsite_csv,
            onsite_img_dir=onsite_img_dir,
            output_csv=f"/content/task2_2_cb_submission_{backbone}.csv"
        )

    elif task == "3.1_submit":
        generate_kaggle_submission(
            backbone="efficientnet",
            ckpt_path="/content/checkpoints/best_task31_se_focal_efficientnet.pt",
            onsite_csv=onsite_csv,
            onsite_img_dir=onsite_img_dir,
            output_csv="/content/task3_1_se_submission_efficientnet.csv",
            use_se=True,
            se_reduction=16
        )

    elif task == "3.2_submit":
        generate_kaggle_submission(
            backbone="efficientnet",
            ckpt_path="/content/checkpoints/best_task32_mha_focal_efficientnet.pt",
            onsite_csv=onsite_csv,
            onsite_img_dir=onsite_img_dir,
            output_csv="/content/task3_2_mha_submission_efficientnet.csv",
            use_mha=True,
            mha_heads=8,
            mha_dropout=0.0
        )

    elif task == "4.1_submit":
        backbone = "swin_t"
        ckpt_path = "/content/checkpoints/best_task41_swin_focal.pt"
        generate_kaggle_submission(
            backbone=backbone,
            ckpt_path=ckpt_path,
            onsite_csv=onsite_csv,
            onsite_img_dir=onsite_img_dir,
            output_csv="/content/task4_1_swin_focal_submission.csv",
            img_size=224,
            threshold=0.5
        )

    elif task == "4.3_submit":
        models_spec = [
            {
                "backbone": "efficientnet",
                "ckpt_path": "/content/checkpoints/best_task21_focal_full_efficientnet.pt",
                "use_se": False,
                "use_mha": False
            },
            {
                "backbone": "efficientnet",
                "ckpt_path": "/content/checkpoints/best_task31_se_focal_efficientnet.pt",
                "use_se": True,
                "se_reduction": 16,
                "use_mha": False
            },
            {
                "backbone": "efficientnet",
                "ckpt_path": "/content/checkpoints/best_task32_mha_focal_efficientnet.pt",
                "use_se": False,
                "use_mha": True,
                "mha_heads": 8,
                "mha_dropout": 0.0
            },
        ]

        weights = [0.50, 0.25, 0.25]

        generate_kaggle_submission_ensemble(
            models_spec=models_spec,
            weights=weights,
            onsite_csv=onsite_csv,
            onsite_img_dir=onsite_img_dir,
            output_csv="/content/task4_3_ensemble_submission.csv",
            img_size=256,
            threshold=0.5
        )

    else:
        raise ValueError("Unknown task. Use: 1.1,1.2,1.3,2.1,2.2,3.1,3.2,4.1 and *_submit variants.")



=== Task 1.1 (No fine-tuning) | backbone=efficientnet | device=cpu ===

DR -> Acc: 0.6000 | Prec: 0.7459 | Rec: 0.6500 | F1: 0.6947 | Kappa: 0.1228
Glaucoma -> Acc: 0.7950 | Prec: 0.5769 | Rec: 0.6122 | F1: 0.5941 | Kappa: 0.4571
AMD -> Acc: 0.7150 | Prec: 0.2464 | Rec: 0.7727 | F1: 0.3736 | Kappa: 0.2482


In [5]:
!ls /content

drive  sample_data


In [37]:
from google.colab import files
files.download("task4_3_ensemble_submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>